In [3]:
import os
import json
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer
import h5py
from tqdm.auto import tqdm

class EEGTextMetaDataset(Dataset):
    def __init__(self, eeg_dir, metadata_dir, tokenizer, 
                 color_map_file, object_map_file,  # <-- ADDED ARGUMENTS
                 max_length=64, use_emotional_tone=False):
        
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.use_emotional_tone = use_emotional_tone

        # -------------------------
        # 1. Load EEG files (all subjects)
        # -------------------------
        eeg_files = []
        for root, dirs, files in os.walk(eeg_dir):
            for f in files:
                if f.endswith(".npy") and "_preprocessed" in f:
                    eeg_files.append(os.path.join(root, f))
        eeg_files = sorted(eeg_files)

        if not eeg_files:
            raise FileNotFoundError(f"No EEG .npy files found in {eeg_dir}")

        self.eeg_file_paths = eeg_files
        self.eeg_data_list = []
        self.index_map = []

        for subj_idx, path in enumerate(self.eeg_file_paths):
            eeg = np.load(path, mmap_mode='r')
            assert eeg.ndim == 3 and eeg.shape[1:] == (62, 400), \
                f"EEG file {path} has shape {eeg.shape}, expected (*, 62, 400)"
            self.eeg_data_list.append(eeg)
            n_samples = eeg.shape[0]
            self.index_map.extend([(subj_idx, i) for i in range(n_samples)])

        total_samples = len(self.index_map)
        print(f"Found {len(self.eeg_file_paths)} EEG files → Total samples: {total_samples}")

        # -------------------------
        # 2. Load Metadata JSONs (WITH FLEXIBLE CAPTION LOADING)
        # -------------------------
        metadata_files = sorted(
            [os.path.join(dp, f)
             for dp, dn, filenames in os.walk(metadata_dir)
             for f in filenames if f.endswith(".json")]
        )
        if not metadata_files:
            raise FileNotFoundError(f"No metadata JSON files found in {metadata_dir}")

        self.metadata_list = []
        for fpath in metadata_files:
            try:
                with open(fpath, 'r', encoding='utf-8') as f:
                    meta = json.load(f)

                # --- NEW FLEXIBLE CAPTION CHECK ---
                caption_data = meta.get("caption")
                caption_text = None # Initialize

                if isinstance(caption_data, str):
                    # Structure 1: { "caption": "This is a string." }
                    caption_text = caption_data
                elif isinstance(caption_data, dict):
                    # Structure 2: { "caption": { "text": "This is a string." } }
                    caption_text = caption_data.get("text")
                
                # Now check if we actually got a valid string
                if not caption_text or not isinstance(caption_text, str):
                    print(f"Warning: Skipping {fpath}, missing, null, or invalid 'caption' structure.")
                    continue # Skip this file
                
                # --- DATA NORMALIZATION ---
                meta['caption'] = caption_text
                
                # --- Existing Color/Object Checks (Replaced assert with a skip) ---
                if "visual_attributes" not in meta or "major_colors" not in meta["visual_attributes"]:
                    print(f"Warning: Skipping {fpath}, missing 'visual_attributes' or 'major_colors'.")
                    continue # Skip this file
                
                if "objects" not in meta.get("semantic_features", {}):
                    meta.setdefault("semantic_features", {})["objects"] = []
                
                if not meta["visual_attributes"].get("major_colors"):
                    print(f"Warning: Missing 'major_colors' list in {fpath}. Assigning default 'black'.")
                    meta["visual_attributes"]["major_colors"] = [{"color": "black"}]

                self.metadata_list.append(meta) # Only append if all checks pass
            
            except json.JSONDecodeError:
                print(f"Warning: Skipping {fpath}, file is not valid JSON.")
                continue
            except Exception as e:
                print(f"Warning: Skipping {fpath} due to unexpected error: {e}")
                continue

        base_count = len(self.metadata_list)
        print(f"Loaded {base_count} metadata JSON files")
        
        if base_count == 0:
             raise ValueError("No valid metadata files were loaded. Check JSON content and paths.")

        # -------------------------
        # 3. Build base captions
        # -------------------------
        base_captions = []
        for meta in self.metadata_list:
            caption_text = meta["caption"] 
            base_captions.append(caption_text)

        # Repeat for each subject
        num_subjects = len(self.eeg_file_paths)
        self.captions = base_captions * num_subjects
        self.metadata_repeated = self.metadata_list * num_subjects
        
        if len(self.index_map) != len(self.captions):
              raise ValueError(f"Data mismatch: Found {len(self.index_map)} EEG samples but {len(self.captions)} captions after processing. Check if 'n_samples' in your EEG files matches the number of JSON files.")

        assert len(self.captions) == len(self.metadata_repeated) == len(self.index_map), \
            "Mismatch after repeating captions and metadata for subjects"

        # -------------------------
        # 4. Load PRE-DEFINED metadata vocabularies (REPLACED AND FIXED)
        # -------------------------
        
        print(f"Loading pre-defined vocabularies from files...")
        
        # Load the color map JSON we created
        try:
            with open(color_map_file, 'r', encoding='utf-8') as f:
                # The JSON file is { "0": "black", "1": "blue", ... }
                # We convert the JSON string keys "0", "1" to integer keys.
                self.id_to_color = {int(k): v for k, v in json.load(f).items()}
        except FileNotFoundError:
            print(f"Error: Color mapping file not found at {color_map_file}")
            raise
        
        # Load the object map JSON (assuming it has the same format)
        try:
            with open(object_map_file, 'r', encoding='utf-8') as f:
                self.id_to_object = {int(k): v for k, v in json.load(f).items()}
        except FileNotFoundError:
            print(f"Error: Object mapping file not found at {object_map_file}")
            raise

        # Now create the reverse mapping (name -> id)
        self.color_to_id = {v: k for k, v in self.id_to_color.items()}
        self.object_to_id = {v: k for k, v in self.id_to_object.items()}

        # Add a "black" color if it wasn't present, just in case
        if "black" not in self.color_to_id:
            new_id = len(self.color_to_id)
            self.color_to_id["black"] = new_id
            self.id_to_color[new_id] = "black"
            print("Warning: 'black' not in color map. Added it as fallback.")

        print(f"Loaded {len(self.color_to_id)} colors | Loaded {len(self.object_to_id)} objects")
    
    def __len__(self):
        return len(self.index_map)

    def __getitem__(self, idx):
        subj_idx, local_idx = self.index_map[idx]
        eeg_tensor = torch.tensor(self.eeg_data_list[subj_idx][local_idx], dtype=torch.float32)
        
        caption = self.captions[idx] 

        # --- Defensive check for string ---
        if not isinstance(caption, str):
            print(f"Warning: Found invalid caption (None?) at index {idx}. Replacing with empty string.")
            caption = "" # Use an empty string instead of None
        
        tokenized = self.tokenizer(
            caption, padding='max_length', truncation=True, max_length=self.max_length, return_tensors="pt"
        )
        
        input_ids = tokenized['input_ids'].squeeze(0)
        
        meta = self.metadata_repeated[idx]
        
        # --- Defensive color loading ---
        try:
            color_name = meta["visual_attributes"]["major_colors"][0]["color"]
            if color_name not in self.color_to_id:
                # This color might exist in the file but was skipped during init (e.g., from an empty file)
                # or wasn't in our final map
                color_name = "black" # Fallback to default
        except (IndexError, KeyError, TypeError):
            color_name = "black" # Fallback to default
            
        color_id = self.color_to_id[color_name]
        
        scalar_meta_tensor = torch.tensor([color_id], dtype=torch.float32)

        objects_present = meta.get("semantic_features", {}).get("objects", [])
        object_multi_hot = torch.zeros(len(self.object_to_id), dtype=torch.float32)
        for obj in objects_present:
            if obj in self.object_to_id:
                obj_id = self.object_to_id[obj]
                object_multi_hot[obj_id] = 1.0
        
        metadata_tensor = torch.cat((scalar_meta_tensor, object_multi_hot))
        
        return eeg_tensor, input_ids, metadata_tensor

In [5]:
import os
import json
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer
import h5py
from tqdm.auto import tqdm

# --- Paths (Set by user) ---
HDF5_FILE = "/home/poorna/data/eeg_dataset_with_qwen.h5"
EEG_DIR = "/home/poorna/data/preprocessed_eeg"
METADATA_DIR = "/home/poorna/projects/video-captioner/jsons"
TOKENIZER_PATH = "/home/poorna/models/bert-base-uncased"

# --- Paths to your mapping files ---
# You mentioned these are in the data directory
COLOR_MAP_FILE = "/home/poorna/data/color_id_to_name_qwen.json"
OBJECT_MAP_FILE = "/home/poorna/data/object_id_to_name_qwen.json" #<-- Please double-check this filename

# --- Dataset ---
print("Initializing dataset...")
tokenizer = BertTokenizer.from_pretrained(TOKENIZER_PATH)

# --- THIS IS THE FIXED LINE ---
# We now pass the required map_file arguments to the constructor
dataset = EEGTextMetaDataset(
    eeg_dir=EEG_DIR,
    metadata_dir=METADATA_DIR,
    tokenizer=tokenizer,
    color_map_file=COLOR_MAP_FILE,
    object_map_file=OBJECT_MAP_FILE
)

loader = DataLoader(dataset, batch_size=1, shuffle=False) # batch=1 for sequential save

# --- Get dynamic shapes from a sample ---
n_samples = len(dataset)

if n_samples == 0:
    print("Error: Dataset is empty. Check your file paths and JSON file content.")
else:
    sample_eeg, sample_input_ids, sample_meta = dataset[0]

    # --- Create HDF5 file ---
    print(f"Creating HDF5 file at {HDF5_FILE}...")
    with h5py.File(HDF5_FILE, "w") as f:
        eeg_shape = (n_samples, *sample_eeg.shape)
        token_shape = (n_samples, *sample_input_ids.shape)
        meta_shape = (n_samples, *sample_meta.shape)

        print(f"Allocating space for {n_samples} samples...")
        print(f"  - EEG shape: {eeg_shape}")
        print(f"  - Token shape: {token_shape}")
        print(f"  - Metadata shape: {meta_shape}")

        eeg_ds = f.create_dataset("eeg", shape=eeg_shape, dtype="float32")
        tokens_ds = f.create_dataset("input_ids", shape=token_shape, dtype="int64")
        meta_ds = f.create_dataset("metadata", shape=meta_shape, dtype="float32")

        # Iterate and save directly to file (low memory usage)
        print("Writing data to HDF5 file...")
        try:
            for idx, (eeg_tensor, input_ids_tensor, metadata_tensor) in enumerate(tqdm(loader, desc="Saving to HDF5")):
                # The squeeze(0) is needed because the DataLoader adds a batch dimension of 1
                eeg_ds[idx] = eeg_tensor.squeeze(0).numpy()
                tokens_ds[idx] = input_ids_tensor.squeeze(0).numpy()
                meta_ds[idx] = metadata_tensor.squeeze(0).numpy()
            
            print(f"\nSuccessfully saved dataset to {HDF5_FILE}")

        except Exception as e:
            print(f"\n--- ERROR during HDF5 writing at index {idx} ---")
            print(f"Error: {e}")
            print("The HDF5 file may be incomplete or corrupted.")

Initializing dataset...
Found 20 EEG files → Total samples: 28000
Loaded 1400 metadata JSON files
Loading pre-defined vocabularies from files...
Loaded 12 colors | Loaded 90 objects
Creating HDF5 file at /home/poorna/data/eeg_dataset_with_qwen.h5...
Allocating space for 28000 samples...
  - EEG shape: (28000, 62, 400)
  - Token shape: (28000, 64)
  - Metadata shape: (28000, 91)
Writing data to HDF5 file...


Saving to HDF5:   0%|          | 0/28000 [00:00<?, ?it/s]


Successfully saved dataset to /home/poorna/data/eeg_dataset_with_qwen.h5


In [6]:
import h5py
import torch

with h5py.File("/home/poorna/data/eeg_dataset_with_qwen.h5", "r") as f:
    print(list(f.keys()))  # ['eeg', 'input_ids', 'metadata']
    eeg_sample = torch.tensor(f["eeg"][0])         # first sample EEG tensor
    tokens_sample = torch.tensor(f["input_ids"][0])
    meta_sample = torch.tensor(f["metadata"][0])

print(eeg_sample.shape, tokens_sample.shape, meta_sample.shape)

['eeg', 'input_ids', 'metadata']
torch.Size([62, 400]) torch.Size([64]) torch.Size([91])


In [8]:
import h5py

HDF5_FILE = "/home/poorna/data/eeg_dataset_with_qwen.h5"

with h5py.File(HDF5_FILE, "r") as f:
    print("\nDatasets in file:", list(f.keys()))

    for name in f.keys():
        dset = f[name]
        print(f"{name}: shape={dset.shape}, dtype={dset.dtype}")

    # Optional: verify a few entries
    sample_idx = 0
    print("\nSample check:")
    print("EEG sample shape:", f["eeg"][sample_idx].shape)
    print("Tokens sample shape:", f["input_ids"][sample_idx].shape)
    print("Metadata sample shape:", f["metadata"][sample_idx].shape)


Datasets in file: ['eeg', 'input_ids', 'metadata']
eeg: shape=(28000, 62, 400), dtype=float32
input_ids: shape=(28000, 64), dtype=int64
metadata: shape=(28000, 91), dtype=float32

Sample check:
EEG sample shape: (62, 400)
Tokens sample shape: (64,)
Metadata sample shape: (91,)
